# Notebook 15: Mixed Precision Training

## 📚 Historical Context

**Timeline**: 2017-2023 - From FP32 to FP16/BF16 to INT8

**The Evolution of Numerical Precision**:
- **Pre-2017**: Everything in FP32 (32-bit floating point)
  - Standard: Safe, stable, but memory-hungry
  - Problem: GPUs had specialized FP16 hardware sitting unused
  - Memory: Major bottleneck for large models

- **2017**: NVIDIA Volta GPUs + Tensor Cores
  - Hardware breakthrough: FP16 ops are 2-8x faster
  - Problem: Pure FP16 training is unstable (numerical underflow)
  - Solution needed: Mixed precision approach

- **2018**: Mixed Precision Training (Micikevicius et al., NVIDIA)
  - Key innovation: Keep master weights in FP32, compute in FP16
  - Loss scaling to prevent gradient underflow
  - Result: 2-3x speedup + 2x memory savings
  - Adopted by: PyTorch, TensorFlow, all major frameworks

- **2019**: Automatic Mixed Precision (AMP)
  - PyTorch 1.6: Built-in AMP with torch.cuda.amp
  - Automatic: Framework decides what to run in FP16 vs FP32
  - Developer-friendly: ~5 lines of code

- **2020**: BF16 (Brain Float 16) - Google TPUs
  - Innovation: Same exponent range as FP32, reduced mantissa
  - Advantage: More stable than FP16, no loss scaling needed
  - Used in: Google's models, TPUs, newer GPUs (A100, H100)

- **2023+**: INT8 and beyond
  - 8-bit training (LLM.int8(), QLoRA)
  - 4-bit quantization
  - Model compression for deployment

**Why This Matters Now**:

**Model size explosion**:
```
2018: BERT-base (110M params) → 1.7GB in FP32
2020: GPT-3 (175B params) → 700GB in FP32
2023: Llama-70B (70B params) → 280GB in FP32

With FP16:
BERT → 0.85GB (fits on most GPUs)
Llama-70B → 140GB (fits on 2x A100-80GB)
```

**Without mixed precision**: Can't train large models at all.
**With mixed precision**: 2x memory, 2-3x speed, same accuracy.

## 🎯 What You'll Learn

1. **Number Representations** - FP32 vs FP16 vs BF16
2. **Why FP16 Fails** - Underflow, overflow, precision loss
3. **Mixed Precision Technique** - Master weights + gradient scaling
4. **PyTorch AMP** - Automatic mixed precision API
5. **Memory Savings** - Before/after benchmarks
6. **Speed Improvements** - Training time comparison
7. **Numerical Stability** - Avoiding NaN and convergence issues

## 💡 Why This Matters

Mixed precision is ESSENTIAL because:
- **Required for large models**: Llama-7B+ won't fit in memory without it
- **2x memory savings**: Train with 2x batch size or 2x model size
- **2-3x speed**: Faster iterations, more experiments
- **Industry standard**: All production training uses it
- **Free improvement**: Minimal code changes, huge gains

**Real example**: Fine-tuning Llama-7B on a single A100-40GB:
- FP32: OOM (out of memory) with batch size 1
- FP16: Works with batch size 4, trains 2.5x faster
- Difference: Between impossible and practical

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"PyTorch Version: {torch.__version__}")
    
    # Check if GPU supports FP16
    compute_capability = torch.cuda.get_device_capability(0)
    supports_fp16 = compute_capability[0] >= 7  # Volta and newer
    supports_bf16 = compute_capability[0] >= 8  # Ampere and newer
    
    print(f"\nPrecision Support:")
    print(f"  FP16 (Half): {supports_fp16}")
    print(f"  BF16 (BFloat16): {supports_bf16}")
else:
    print("\nWarning: CUDA not available. Mixed precision benefits are GPU-only.")
    print("Code will still run but without speed/memory improvements.")

## Part 1: Understanding Number Representations

### Floating Point Formats:

**FP32 (Float32)** - Standard precision:
```
32 bits total:
  1 bit:  Sign
  8 bits: Exponent
  23 bits: Mantissa (precision)

Range:    ~1.4e-45 to ~3.4e38
Precision: ~7 decimal digits
```

**FP16 (Float16 / Half)** - Reduced precision:
```
16 bits total:
  1 bit:  Sign
  5 bits: Exponent
  10 bits: Mantissa

Range:    ~6e-8 to ~6.5e4
Precision: ~3 decimal digits
```

**BF16 (BFloat16)** - Balanced format:
```
16 bits total:
  1 bit:  Sign
  8 bits: Exponent (same as FP32!)
  7 bits: Mantissa

Range:    Same as FP32 (~1.4e-45 to ~3.4e38)
Precision: ~2 decimal digits
```

### Why FP16 is Problematic:

**Problem 1: Limited range**
```python
# Gradient underflow
gradient = 1e-8  # Small but important gradient
gradient_fp16 = torch.tensor(gradient, dtype=torch.float16)
# Result: 0.0 (underflows to zero!)
```

**Problem 2: Overflow**
```python
# Large activation
activation = 1e5
activation_fp16 = torch.tensor(activation, dtype=torch.float16)
# Result: inf (overflow!)
```

**Problem 3: Precision loss**
```python
# Weight update
weight = 1.0
gradient = 1e-4  # Small update
weight_fp16 = weight - gradient  # In FP16
# Result: 1.0 (update lost due to precision!)
```

### BF16 Advantages:

**Same range as FP32**:
- No underflow/overflow issues
- No loss scaling needed
- Simpler training code

**Trade-off**:
- Less precision than FP16
- But range matters more than precision for training

### Visual Comparison:

In [ ]:
def visualize_precision_formats():
    """
    Visualize differences between FP32, FP16, and BF16.
    """
    # Test values covering different ranges
    test_values = np.array([
        1e-8, 1e-6, 1e-4, 1e-2, 1e-1,  # Small values
        1.0, 10.0, 100.0,               # Medium values
        1e3, 1e4, 1e5                   # Large values
    ])
    
    # Convert to different precisions
    fp32_values = torch.tensor(test_values, dtype=torch.float32)
    fp16_values = torch.tensor(test_values, dtype=torch.float16)
    bf16_values = torch.tensor(test_values, dtype=torch.bfloat16) if supports_bf16 else None
    
    # Compute relative errors
    fp16_errors = torch.abs(fp32_values - fp16_values.float()) / (fp32_values + 1e-10)
    if bf16_values is not None:
        bf16_errors = torch.abs(fp32_values - bf16_values.float()) / (fp32_values + 1e-10)
    
    # Create comparison table
    print("="*80)
    print("PRECISION COMPARISON")
    print("="*80)
    print(f"{'Value':<12} {'FP32':<15} {'FP16':<15} {'FP16 Error':<12} {'Notes':<20}")
    print("-"*80)
    
    for i, val in enumerate(test_values):
        fp32_str = f"{fp32_values[i].item():.6e}"
        fp16_str = f"{fp16_values[i].item():.6e}"
        error_str = f"{fp16_errors[i].item():.2%}"
        
        # Add notes about issues
        notes = ""
        if fp16_values[i].item() == 0 and fp32_values[i].item() != 0:
            notes = "UNDERFLOW!"
        elif torch.isinf(fp16_values[i]):
            notes = "OVERFLOW!"
        elif fp16_errors[i] > 0.01:
            notes = "High error"
        
        print(f"{val:<12.0e} {fp32_str:<15} {fp16_str:<15} {error_str:<12} {notes:<20}")
    
    print("="*80)
    
    # Visualize range comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Representation range
    formats = ['FP32', 'FP16', 'BF16']
    min_vals = [1.4e-45, 6e-8, 1.4e-45]
    max_vals = [3.4e38, 6.5e4, 3.4e38]
    precisions = [23, 10, 7]
    
    axes[0].barh(formats, np.log10(max_vals), color=['blue', 'orange', 'green'], alpha=0.7)
    axes[0].set_xlabel('Max Value (log10 scale)')
    axes[0].set_title('Representable Range')
    axes[0].grid(True, alpha=0.3, axis='x')
    
    # Plot 2: Precision bits
    axes[1].bar(formats, precisions, color=['blue', 'orange', 'green'], alpha=0.7)
    axes[1].set_ylabel('Mantissa Bits (Precision)')
    axes[1].set_title('Precision Comparison')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("\nKey Observations:")
    print("1. FP16 has much smaller range → underflow/overflow risk")
    print("2. BF16 has same range as FP32 → more stable")
    print("3. Small gradients (<1e-7) become zero in FP16")
    print("4. Large activations (>65000) overflow in FP16")
    print("5. This is why we need special techniques for FP16 training")

visualize_precision_formats()

## Part 2: Mixed Precision Training Technique

### The Core Idea:

**Keep master weights in FP32, compute in FP16**:

```python
# Pseudocode for mixed precision training

weights_fp32 = model.parameters()  # Master weights in FP32

for batch in dataloader:
    # 1. Forward pass in FP16 (fast)
    weights_fp16 = weights_fp32.half()
    inputs_fp16 = inputs.half()
    outputs = model(inputs_fp16)  # Uses FP16
    loss = criterion(outputs, labels)
    
    # 2. Loss scaling (prevent gradient underflow)
    scaled_loss = loss * scale_factor  # e.g., 65536
    
    # 3. Backward pass in FP16
    scaled_loss.backward()  # Gradients in FP16
    
    # 4. Unscale gradients and update FP32 master weights
    gradients = [p.grad / scale_factor for p in weights_fp32]
    optimizer.step()  # Updates FP32 weights
```

### Loss Scaling - Critical for FP16:

**Problem**: Small gradients underflow to zero in FP16
```
gradient = 1e-8  # Important but small
gradient_fp16 = fp16(1e-8) = 0.0  # Lost!
```

**Solution**: Scale loss before backward pass
```
loss_scaled = loss * 65536  # Large scale factor
gradient_scaled = 1e-8 * 65536 = 0.00065  # Representable in FP16!
gradient = gradient_scaled / 65536  # Unscale in FP32
```

**Dynamic Loss Scaling**:
- Start with large scale (e.g., 65536)
- If overflow detected (NaN/Inf) → reduce scale by 2x
- If no overflow for N steps → increase scale by 2x
- Automatically adapts to your model

### PyTorch AMP Components:

**1. autocast**: Automatically casts operations to FP16
```python
with torch.cuda.amp.autocast():
    outputs = model(inputs)  # Runs in FP16
    loss = criterion(outputs, labels)
```

**2. GradScaler**: Handles loss scaling automatically
```python
scaler = GradScaler()
scaler.scale(loss).backward()  # Scaled backward
scaler.step(optimizer)         # Unscale and update
scaler.update()                # Adjust scale factor
```

### What Stays FP32:

**Critical operations that need precision**:
- Layer normalization
- Softmax
- Loss computation
- Batch normalization
- Master weight storage

**PyTorch AMP handles this automatically** - you don't need to think about it!

In [ ]:
def demonstrate_loss_scaling():
    """
    Demonstrate why loss scaling is necessary.
    """
    print("\n" + "="*60)
    print("DEMONSTRATION: LOSS SCALING")
    print("="*60)
    
    # Simulate small gradient
    gradient_fp32 = torch.tensor([1e-8, 1e-7, 1e-6, 1e-5, 1e-4], dtype=torch.float32)
    
    print("\nOriginal gradients (FP32):")
    print(gradient_fp32)
    
    # Convert to FP16 without scaling
    gradient_fp16_no_scale = gradient_fp32.half()
    print("\nConverted to FP16 (no scaling):")
    print(gradient_fp16_no_scale)
    print("Note: Smallest gradients became ZERO!")
    
    # With loss scaling
    scale_factor = 65536
    gradient_scaled = gradient_fp32 * scale_factor
    gradient_fp16_scaled = gradient_scaled.half()
    gradient_unscaled = (gradient_fp16_scaled.float() / scale_factor)
    
    print(f"\nWith loss scaling (scale={scale_factor}):")
    print("Scaled gradients (FP16):")
    print(gradient_fp16_scaled)
    print("\nUnscaled back to FP32:")
    print(gradient_unscaled)
    print("Note: Gradients preserved!")
    
    # Compute errors
    error_no_scale = torch.abs(gradient_fp32 - gradient_fp16_no_scale.float())
    error_with_scale = torch.abs(gradient_fp32 - gradient_unscaled)
    
    print("\n" + "="*60)
    print("COMPARISON")
    print("="*60)
    print(f"{'Gradient':<12} {'No Scaling':<15} {'With Scaling':<15}")
    print("-"*60)
    for i in range(len(gradient_fp32)):
        print(f"{gradient_fp32[i].item():.1e}      "
              f"{error_no_scale[i].item():.1e}         "
              f"{error_with_scale[i].item():.1e}")
    print("="*60)
    print("\nConclusion: Loss scaling is ESSENTIAL for FP16 training!")
    print("It prevents gradient underflow and preserves small updates.")

demonstrate_loss_scaling()

## Part 3: Implementing Mixed Precision Training

### Standard FP32 Training (Baseline):

```python
# Standard training loop
for inputs, labels in dataloader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
```

### Mixed Precision Training with AMP:

```python
# Mixed precision training loop
scaler = GradScaler()

for inputs, labels in dataloader:
    optimizer.zero_grad()
    
    # Forward pass in FP16
    with autocast():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
    
    # Scaled backward pass
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
```

**That's it! Just 5 additional lines of code.**

### Let's compare both on a real task:

In [ ]:
# Create a reasonably sized model for testing
class MediumModel(nn.Module):
    """
    Medium-sized model to demonstrate memory/speed benefits.
    Similar complexity to small transformers.
    """
    def __init__(self, input_dim=128, hidden_dim=512, num_classes=10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x):
        return self.layers(x)

# Simple dataset
class SyntheticDataset(Dataset):
    def __init__(self, num_samples=10000, input_dim=128, num_classes=10):
        self.data = torch.randn(num_samples, input_dim)
        self.labels = torch.randint(0, num_classes, (num_samples,))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

def get_model_memory(model):
    """Calculate model memory usage."""
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / 1024**2  # Convert to MB

def train_epoch(model, dataloader, criterion, optimizer, use_amp=False):
    """
    Train for one epoch.
    
    Args:
        use_amp: If True, use automatic mixed precision
    
    Returns:
        avg_loss: Average loss for epoch
        time_taken: Time in seconds
        peak_memory: Peak GPU memory in MB
    """
    model.train()
    total_loss = 0.0
    
    # Initialize scaler if using AMP
    scaler = GradScaler() if use_amp else None
    
    # Reset peak memory stats
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    start_time = time.time()
    
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            # Mixed precision training
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            # Standard FP32 training
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
    
    end_time = time.time()
    
    avg_loss = total_loss / len(dataloader)
    time_taken = end_time - start_time
    
    # Get peak memory
    peak_memory = 0.0
    if torch.cuda.is_available():
        peak_memory = torch.cuda.max_memory_allocated() / 1024**2  # MB
    
    return avg_loss, time_taken, peak_memory

# Setup
print("\n" + "="*60)
print("EXPERIMENT: FP32 vs MIXED PRECISION (FP16)")
print("="*60)

# Create dataset
train_dataset = SyntheticDataset(num_samples=10000)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

# Model info
test_model = MediumModel().to(device)
num_params = sum(p.numel() for p in test_model.parameters())
model_memory = get_model_memory(test_model)

print(f"\nModel Information:")
print(f"  Parameters: {num_params:,}")
print(f"  Model size (FP32): {model_memory:.2f} MB")
print(f"  Model size (FP16): {model_memory/2:.2f} MB (expected)")
print(f"  Batch size: 256")
print(f"  Training samples: 10,000")

In [ ]:
# Train with FP32
print("\n" + "="*60)
print("TRAINING WITH FP32 (Baseline)")
print("="*60)

model_fp32 = MediumModel().to(device)
optimizer_fp32 = torch.optim.AdamW(model_fp32.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

fp32_results = {'loss': [], 'time': [], 'memory': []}

for epoch in range(5):
    loss, time_taken, peak_memory = train_epoch(
        model_fp32, train_loader, criterion, optimizer_fp32, use_amp=False
    )
    fp32_results['loss'].append(loss)
    fp32_results['time'].append(time_taken)
    fp32_results['memory'].append(peak_memory)
    
    print(f"Epoch {epoch+1}/5: Loss={loss:.4f}, "
          f"Time={time_taken:.2f}s, Memory={peak_memory:.1f}MB")

avg_time_fp32 = np.mean(fp32_results['time'])
avg_memory_fp32 = np.mean(fp32_results['memory'])

print(f"\nFP32 Average: Time={avg_time_fp32:.2f}s, Memory={avg_memory_fp32:.1f}MB")

In [ ]:
# Train with Mixed Precision
print("\n" + "="*60)
print("TRAINING WITH MIXED PRECISION (FP16 + AMP)")
print("="*60)

model_amp = MediumModel().to(device)
optimizer_amp = torch.optim.AdamW(model_amp.parameters(), lr=1e-3)

amp_results = {'loss': [], 'time': [], 'memory': []}

for epoch in range(5):
    loss, time_taken, peak_memory = train_epoch(
        model_amp, train_loader, criterion, optimizer_amp, use_amp=True
    )
    amp_results['loss'].append(loss)
    amp_results['time'].append(time_taken)
    amp_results['memory'].append(peak_memory)
    
    print(f"Epoch {epoch+1}/5: Loss={loss:.4f}, "
          f"Time={time_taken:.2f}s, Memory={peak_memory:.1f}MB")

avg_time_amp = np.mean(amp_results['time'])
avg_memory_amp = np.mean(amp_results['memory'])

print(f"\nAMP Average: Time={avg_time_amp:.2f}s, Memory={avg_memory_amp:.1f}MB")

# Comparison
print("\n" + "="*60)
print("COMPARISON RESULTS")
print("="*60)
print(f"{'Metric':<20} {'FP32':<15} {'Mixed (FP16)':<15} {'Improvement':<15}")
print("-"*60)

speedup = avg_time_fp32 / avg_time_amp
memory_savings = (avg_memory_fp32 - avg_memory_amp) / avg_memory_fp32 * 100

print(f"{'Time per epoch':<20} {avg_time_fp32:<15.2f} {avg_time_amp:<15.2f} {speedup:.2f}x faster")
print(f"{'Peak memory (MB)':<20} {avg_memory_fp32:<15.1f} {avg_memory_amp:<15.1f} {memory_savings:.1f}% less")
print(f"{'Final loss':<20} {fp32_results['loss'][-1]:<15.4f} {amp_results['loss'][-1]:<15.4f} ~Same")

print("\n" + "="*60)
print("KEY INSIGHTS:")
print(f"1. Speed improvement: {speedup:.2f}x faster with mixed precision")
print(f"2. Memory savings: {memory_savings:.1f}% reduction")
print("3. Loss convergence: Virtually identical")
print("4. Implementation: Only 5 extra lines of code!")
print("="*60)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Training time
epochs = range(1, 6)
axes[0].plot(epochs, fp32_results['time'], marker='o', label='FP32', linewidth=2)
axes[0].plot(epochs, amp_results['time'], marker='s', label='Mixed (FP16)', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Time (seconds)')
axes[0].set_title('Training Time Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Memory usage
axes[1].plot(epochs, fp32_results['memory'], marker='o', label='FP32', linewidth=2)
axes[1].plot(epochs, amp_results['memory'], marker='s', label='Mixed (FP16)', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Peak Memory (MB)')
axes[1].set_title('Memory Usage Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3: Loss convergence
axes[2].plot(epochs, fp32_results['loss'], marker='o', label='FP32', linewidth=2)
axes[2].plot(epochs, amp_results['loss'], marker='s', label='Mixed (FP16)', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].set_title('Loss Convergence')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nConclusion: Mixed precision provides significant speedup and memory")
print("savings with no loss in model accuracy. It's a free lunch!")

## Part 4: Numerical Stability and Debugging

### Common Issues with Mixed Precision:

**1. NaN/Inf During Training**

**Causes**:
```
- Loss scale too high → gradient overflow
- Unstable operations (exp, log) on large values
- Learning rate too high
- Batch norm with very small batches
```

**Solutions**:
```python
# 1. Use gradient clipping
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
scaler.step(optimizer)

# 2. Lower initial loss scale
scaler = GradScaler(init_scale=2**10)  # Default is 2**16

# 3. Add epsilon to log operations
loss = -torch.log(pred + 1e-7)  # Prevent log(0)
```

**2. Model Doesn't Converge**

**Causes**:
```
- Some operations not benefiting from FP16
- Loss scale too aggressive
- Precision-sensitive operations in FP16
```

**Solutions**:
```python
# Force specific operations to FP32
with autocast(enabled=False):
    # Critical computation in FP32
    output = precise_operation(input.float())
```

**3. Slower Than Expected**

**Causes**:
```
- GPU doesn't support FP16 tensor cores
- Batch size too small (no benefit)
- Model too small (overhead dominates)
- CPU operations in the loop
```

**Requirements for speedup**:
```
- GPU: Volta (V100) or newer
- Batch size: ≥32 (larger is better)
- Model size: ≥10M parameters
```

### Debugging Checklist:

```python
# Check for NaN/Inf
if torch.isnan(loss) or torch.isinf(loss):
    print("Loss is NaN or Inf!")
    # Reduce loss scale or clip gradients

# Monitor gradient norms
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
print(f"Gradient norm: {grad_norm}")
# If very large (>100), use gradient clipping

# Check scaler state
print(f"Current loss scale: {scaler.get_scale()}")
# Should be around 2^10 to 2^16
```

In [ ]:
def train_with_stability_checks(
    model, 
    dataloader, 
    criterion, 
    optimizer, 
    num_epochs=3,
    use_amp=True
):
    """
    Training with comprehensive stability checks.
    
    Demonstrates how to monitor and debug mixed precision training.
    """
    scaler = GradScaler() if use_amp else None
    
    print("\n" + "="*60)
    print("TRAINING WITH STABILITY MONITORING")
    print("="*60)
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        
        # Track statistics
        nan_count = 0
        grad_norms = []
        loss_scales = []
        
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            if use_amp:
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                
                # Check for NaN
                if torch.isnan(loss) or torch.isinf(loss):
                    nan_count += 1
                    print(f"  Warning: NaN/Inf detected at batch {batch_idx}")
                    continue
                
                scaler.scale(loss).backward()
                
                # Unscale for gradient norm computation
                scaler.unscale_(optimizer)
                
                # Compute gradient norm
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )
                grad_norms.append(grad_norm.item())
                
                # Track loss scale
                loss_scales.append(scaler.get_scale())
                
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )
                grad_norms.append(grad_norm.item())
                
                optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(dataloader)
        avg_grad_norm = np.mean(grad_norms)
        
        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Loss: {avg_loss:.4f}")
        print(f"  Avg Gradient Norm: {avg_grad_norm:.4f}")
        
        if use_amp:
            print(f"  Final Loss Scale: {loss_scales[-1]:.0f}")
            print(f"  NaN/Inf count: {nan_count}")
        
        # Warning if gradient norm is too high
        if avg_grad_norm > 10:
            print("  ⚠ Warning: High gradient norms detected!")
            print("    Consider: Lower LR, stronger clipping, or check data")
        
        # Warning if many NaNs
        if nan_count > len(dataloader) * 0.1:
            print("  ⚠ Warning: Frequent NaN/Inf!")
            print("    Consider: Lower loss scale or stronger gradient clipping")
    
    print("\n" + "="*60)

# Test stability monitoring
print("\nDemonstrating stability monitoring with mixed precision...")
model_stable = MediumModel().to(device)
optimizer_stable = torch.optim.AdamW(model_stable.parameters(), lr=1e-3)

train_with_stability_checks(
    model_stable, train_loader, criterion, optimizer_stable,
    num_epochs=3, use_amp=True
)

print("\nStability monitoring helps catch issues early!")
print("In production, log these metrics to wandb/tensorboard.")

## 📊 Summary

### What We Learned:

**1. Number Representations**
- FP32: Standard, 32 bits, stable but memory-hungry
- FP16: Half precision, 16 bits, 2x memory savings but unstable
- BF16: Brain Float, 16 bits, stable with same range as FP32

**2. Why Pure FP16 Fails**
- Gradient underflow: Small gradients become zero
- Overflow: Large values become infinity
- Precision loss: Weight updates lost

**3. Mixed Precision Solution**
- Keep master weights in FP32
- Compute forward/backward in FP16
- Use loss scaling to prevent underflow
- Update FP32 weights with unscaled gradients

**4. PyTorch AMP Implementation**
```python
# Just 5 lines of code!
scaler = GradScaler()

with autocast():
    outputs = model(inputs)
    loss = criterion(outputs, labels)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```

**5. Benefits**
- 2x memory savings → train larger models or bigger batches
- 2-3x speedup → faster iterations and experiments
- Same accuracy → no loss in model quality
- Minimal code changes → easy to adopt

### Decision Guide: When to Use Mixed Precision?

```
GPU Check:
  ├─ Pre-Volta (K80, P100) → FP16 won't help much
  ├─ Volta+ (V100, T4) → FP16 gives 2-3x speedup
  └─ Ampere+ (A100, RTX 30xx) → BF16 preferred if available

Model Size:
  ├─ < 10M params → Benefit small, but still worth it
  ├─ 10M - 1B params → Significant benefit (use it!)
  └─ > 1B params → Essential, won't fit without it

Batch Size:
  ├─ < 8 → Limited benefit
  ├─ 8 - 32 → Moderate benefit
  └─ > 32 → Full benefit
```

### Best Practices:

**1. Always Use Gradient Clipping**
```python
scaler.unscale_(optimizer)
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
scaler.step(optimizer)
```

**2. Monitor for Issues**
- Log loss scale: Should be between 2^10 and 2^16
- Check for NaN/Inf: Should be rare (<1% of batches)
- Track gradient norms: Should be <10 typically

**3. Use BF16 if Available**
```python
# Check GPU support
if torch.cuda.is_bf16_supported():
    with autocast(dtype=torch.bfloat16):
        outputs = model(inputs)
        # More stable, no loss scaling needed
```

**4. Adjust Loss Scale if Needed**
```python
# If getting frequent NaN:
scaler = GradScaler(init_scale=2**10)  # Lower initial scale

# If no overflow detected:
scaler = GradScaler(growth_interval=100)  # Grow scale slower
```

### Real-World Impact:

| Model | FP32 Memory | FP16 Memory | Speedup | Status |
|-------|-------------|-------------|---------|--------|
| BERT-base | 1.7 GB | 0.85 GB | 2.3x | Practical on 8GB GPU |
| Llama-7B | 28 GB | 14 GB | 2.5x | Fits on single A100-40GB |
| Llama-13B | 52 GB | 26 GB | 2.8x | Fits on single A100-80GB |
| Llama-70B | 280 GB | 140 GB | 3.0x | Possible with 2x A100-80GB |

### Common Pitfalls:

**Pitfall 1: Forgetting scaler.update()**
```python
# Wrong:
scaler.scale(loss).backward()
scaler.step(optimizer)
# Missing scaler.update() → loss scale never adjusts!

# Correct:
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()  # Essential!
```

**Pitfall 2: Not unscaling before gradient clipping**
```python
# Wrong:
scaler.scale(loss).backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Clips scaled gradients!
scaler.step(optimizer)

# Correct:
scaler.scale(loss).backward()
scaler.unscale_(optimizer)  # Unscale first!
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
scaler.step(optimizer)
```

**Pitfall 3: Using on CPU**
```python
# Won't benefit on CPU
if torch.cuda.is_available():
    use_amp = True
else:
    use_amp = False
```

### Next Steps:

**Stage 2: Advanced Techniques**
- LoRA (Low-Rank Adaptation) with mixed precision
- QLoRA (4-bit + LoRA)
- Gradient checkpointing + mixed precision
- DeepSpeed ZeRO optimizer

**Stage 3: Production Deployment**
- INT8 quantization for inference
- ONNX export with FP16
- TensorRT optimization
- Model distillation

### Production Checklist:

✓ **Use mixed precision for all training** (unless on old GPUs)
✓ **Prefer BF16 over FP16** (if supported)
✓ **Always gradient clip** with mixed precision
✓ **Monitor loss scale** and gradient norms
✓ **Test convergence** vs FP32 baseline
✓ **Log to wandb** for tracking

---

**Key Takeaway**: Mixed precision is the **easiest** way to get massive improvements in deep learning. Just 5 lines of code gives you 2x memory and 2-3x speed. In 2024, NOT using mixed precision is like leaving free money on the table. Always use it for training large models!